# AIRBUS Filing Strategy Analysis (EPO TIP Environment)

**Created:** 2025-02-06
**Purpose:** Comprehensive analysis of Airbus patent strategy over 10+ years
**Target audience:** PATLIB training, PIZnet seminar, consulting demo
**Execution:** EPO Technology Intelligence Platform (TIP) or BigQuery

---

### Important: Airbus Name Variants in PATSTAT

Airbus has many name variants in PATSTAT:

- **AIRBUS** (parent company)
- **AIRBUS OPERATIONS** (aircraft manufacturing)
- **AIRBUS DEFENCE AND SPACE** (formerly EADS/Cassidian)
- **AIRBUS HELICOPTERS** (formerly Eurocopter)
- **AIRBUS GROUP** (holding name 2014-2017)
- Historical names: **EADS, EUROCOPTER, ASTRIUM, CASSIDIAN**

The queries use `han_name` (harmonized names) **AND** `person_name` for maximum coverage.

### Query Overview

| Query | Topic |
|-------|-------|
| A | Identify name variants |
| B | Overall filing trend (2014-2024) |
| C | Geographic filing strategy |
| D | Technology fields and shifts |
| E | IPC main classes top 25 with trend |
| F | Grant rate and time-to-grant |
| G | Co-applicants and cooperation strategy |
| H | Patent family size as strategy indicator |
| I | Sustainability / future technology patents |
| J | Inventor locations (NUTS regions) |

In [1]:
from epo.tipdata.patstat import PatstatClient
import pandas as pd
import time

# Connect to PATSTAT
patstat = PatstatClient(env='PROD')

def timed_query(query):
    """Execute query and return DataFrame with timing."""
    start = time.time()
    res = patstat.sql_query(query, use_legacy_sql=False)
    print(f"Query took {time.time() - start:.2f}s ({len(res)} rows)")
    return pd.DataFrame(res)

# QUERY A: Identify Airbus Name Variants

### Business Question
Which name variants of Airbus exist in PATSTAT and how many filings does each variant have?

**Stakeholder:** All – foundation for further analyses

**Explanation:** Initial exploration to understand the filing landscape.
Airbus has many variants due to renamings and subsidiaries.

In [2]:
df_a = timed_query("""
SELECT
    p.person_name,
    p.han_name,
    p.psn_name,
    p.psn_sector,
    p.person_ctry_code,
    COUNT(DISTINCT pa.appln_id) AS applications,
    MIN(a.appln_filing_year) AS first_filing_year,
    MAX(a.appln_filing_year) AS last_filing_year
FROM tls206_person p
JOIN tls207_pers_appln pa ON p.person_id = pa.person_id
JOIN tls201_appln a ON pa.appln_id = a.appln_id
WHERE pa.applt_seq_nr > 0
  AND (
    LOWER(p.person_name) LIKE '%airbus%'
    OR LOWER(p.han_name) LIKE '%airbus%'
    OR LOWER(p.person_name) LIKE '%eurocopter%'
    OR LOWER(p.person_name) LIKE '%astrium%'
    OR LOWER(p.person_name) LIKE '%cassidian%'
    OR (LOWER(p.person_name) LIKE '%eads%'
        AND LOWER(p.person_name) NOT LIKE '%beads%'
        AND LOWER(p.person_name) NOT LIKE '%leads%'
        AND LOWER(p.person_name) NOT LIKE '%heads%')
  )
  AND a.appln_filing_year >= 2000
GROUP BY p.person_name, p.han_name, p.psn_name, p.psn_sector, p.person_ctry_code
HAVING COUNT(DISTINCT pa.appln_id) >= 5
ORDER BY applications DESC
LIMIT 100;
""")

df_a.to_csv('airbus_a_namensvarianten.csv', index=False)
df_a

Query took 4.72s (100 rows)


,person_name,han_name,psn_name,psn_sector,person_ctry_code,applications,first_filing_year,last_filing_year
0,Airbus Operations GmbH,AIRBUS OPERATIONS GMBH,AIRBUS OPERATIONS,COMPANY,DE,9730,2001,2025
1,Airbus Deutschland GmbH,AIRBUS OPERATIONS GMBH,AIRBUS DEUTSCHLAND,COMPANY,DE,3553,2000,2013
2,Airbus France,AIRBUS FR,AIRBUS FRANCE,COMPANY,FR,2954,2000,2010
3,AIRBUS OPERATIONS,AIRBUS OPERATIONS SAS,AIRBUS OPERATIONS,COMPANY,FR,2841,2001,2024
4,EADS Deutschland GmbH,EADS DEUTSCHLAND GMBH,EADS DEUTSCHLAND,COMPANY,DE,2741,2000,2017
...,...,...,...,...,...,...,...,...
95,Airbus SAS,AIRBUS SAS,Airbus SAS,UNKNOWN,FR,68,2021,2024
96,AIRBUS (S.A.S.),AIRBUS (S.A.S.),AIRBUS (S.A.S.),COMPANY,,67,2009,2023
97,"Airbus Operations, S.L.U.","Airbus Operations, S.L.U.",AIRBUS OPERATIONS,COMPANY,,66,2015,2022
98,Airbus Defence and Space SA,AIRBUS DEFENCE & SPACE SA,AIRBUS DEFENCE AND SPACE,COMPANY,ES,64,2010,2023


# QUERY B: Airbus Overall Filing Trend (2014-2024)

### Business Question
How has Airbus's overall patent filing activity developed over the last 10 years? Is the portfolio growing, declining, or stagnating?

**Stakeholder:** IP strategy, management, investor communications

**Explanation:** Counts DOCDB families (not individual filings) to avoid double-counting of the same protected right. Distinguishes by business divisions (Operations, Defence, Helicopters).

In [3]:
df_b = timed_query("""
WITH airbus_apps AS (
    SELECT DISTINCT
        a.appln_id,
        a.docdb_family_id,
        a.appln_filing_year,
        a.appln_auth,
        a.granted,
        a.docdb_family_size,
        CASE
            WHEN LOWER(p.person_name) LIKE '%airbus operations%' THEN 'Airbus Operations (Flugzeugbau)'
            WHEN LOWER(p.person_name) LIKE '%airbus defence%'
              OR LOWER(p.person_name) LIKE '%airbus defense%' THEN 'Airbus Defence & Space'
            WHEN LOWER(p.person_name) LIKE '%airbus helicopter%'
              OR LOWER(p.person_name) LIKE '%eurocopter%' THEN 'Airbus Helicopters'
            WHEN LOWER(p.person_name) LIKE '%airbus group%' THEN 'Airbus Group (Holding)'
            WHEN LOWER(p.person_name) LIKE '%airbus s%a%s%' THEN 'Airbus SAS (Zentrale)'
            WHEN LOWER(p.person_name) LIKE '%astrium%' THEN 'Astrium (→ Defence & Space)'
            WHEN LOWER(p.person_name) LIKE '%cassidian%' THEN 'Cassidian (→ Defence & Space)'
            WHEN LOWER(p.person_name) LIKE '%eads%' THEN 'EADS (→ Airbus Group)'
            ELSE 'Airbus (andere/unspezifisch)'
        END AS business_unit
    FROM tls201_appln a
    JOIN tls207_pers_appln pa ON a.appln_id = pa.appln_id
    JOIN tls206_person p ON pa.person_id = p.person_id
    WHERE pa.applt_seq_nr > 0
      AND a.appln_filing_year BETWEEN 2014 AND 2024
      AND (
        LOWER(p.person_name) LIKE '%airbus%'
        OR LOWER(p.person_name) LIKE '%eurocopter%'
        OR LOWER(p.person_name) LIKE '%astrium%'
        OR LOWER(p.person_name) LIKE '%cassidian%'
        OR (LOWER(p.person_name) LIKE '%eads%'
            AND LOWER(p.person_name) NOT LIKE '%beads%'
            AND LOWER(p.person_name) NOT LIKE '%leads%'
            AND LOWER(p.person_name) NOT LIKE '%heads%')
      )
)

SELECT
    appln_filing_year,
    business_unit,
    COUNT(DISTINCT docdb_family_id) AS patent_families,
    COUNT(DISTINCT appln_id) AS total_applications,
    COUNT(DISTINCT CASE WHEN granted = 'Y' THEN appln_id END) AS granted_applications,
    ROUND(AVG(docdb_family_size), 1) AS avg_family_size
FROM airbus_apps
GROUP BY appln_filing_year, business_unit
ORDER BY appln_filing_year, patent_families DESC;
""")

df_b.to_csv('airbus_b_anmeldetrend.csv', index=False)
df_b

Query took 6.03s (81 rows)


,appln_filing_year,business_unit,patent_families,total_applications,granted_applications,avg_family_size
0,2014,Airbus Operations (Flugzeugbau),1029,1422,978,4.2
1,2014,Airbus Defence & Space,290,542,344,5.0
2,2014,Airbus Helicopters,209,325,296,4.0
3,2014,Airbus (andere/unspezifisch),162,222,163,3.9
4,2014,EADS (→ Airbus Group),87,124,67,5.1
...,...,...,...,...,...,...
76,2024,Airbus (andere/unspezifisch),80,93,13,2.9
77,2024,Airbus SAS (Zentrale),80,134,2,3.1
78,2024,Airbus Defence & Space,50,73,3,2.5
79,2024,EADS (→ Airbus Group),49,77,7,3.8


# QUERY C: Geographic Filing Strategy

### Business Question
In which countries/regions does Airbus file its patents? Has the geographic strategy changed (e.g., more filings in China)?

**Stakeholder:** IP portfolio management, international expansion

**Explanation:** Compares filing authorities (`appln_auth`) between two periods: 2014-2018 (early) vs. 2019-2024 (late). Shows shifts in geographic priority.

In [4]:
df_c = timed_query("""
WITH airbus_apps AS (
    SELECT DISTINCT
        a.appln_id,
        a.docdb_family_id,
        a.appln_auth,
        a.appln_filing_year,
        CASE
            WHEN a.appln_filing_year BETWEEN 2014 AND 2018 THEN 'Periode 1 (2014-2018)'
            WHEN a.appln_filing_year BETWEEN 2019 AND 2024 THEN 'Periode 2 (2019-2024)'
        END AS period
    FROM tls201_appln a
    JOIN tls207_pers_appln pa ON a.appln_id = pa.appln_id
    JOIN tls206_person p ON pa.person_id = p.person_id
    WHERE pa.applt_seq_nr > 0
      AND a.appln_filing_year BETWEEN 2014 AND 2024
      AND (
        LOWER(p.person_name) LIKE '%airbus%'
        OR LOWER(p.person_name) LIKE '%eurocopter%'
        OR LOWER(p.person_name) LIKE '%astrium%'
      )
),

geo_by_period AS (
    SELECT
        period,
        appln_auth,
        COUNT(DISTINCT docdb_family_id) AS families,
        COUNT(DISTINCT appln_id) AS applications
    FROM airbus_apps
    WHERE period IS NOT NULL
    GROUP BY period, appln_auth
    HAVING COUNT(DISTINCT appln_id) >= 10
)

SELECT
    appln_auth,
    CASE
        WHEN appln_auth = 'EP' THEN 'Europäisches Patentamt'
        WHEN appln_auth = 'US' THEN 'US Patent Office'
        WHEN appln_auth = 'CN' THEN 'China (CNIPA)'
        WHEN appln_auth = 'FR' THEN 'Frankreich (INPI)'
        WHEN appln_auth = 'DE' THEN 'Deutschland (DPMA)'
        WHEN appln_auth = 'WO' THEN 'PCT (WIPO)'
        WHEN appln_auth = 'JP' THEN 'Japan (JPO)'
        WHEN appln_auth = 'KR' THEN 'Südkorea (KIPO)'
        WHEN appln_auth = 'IN' THEN 'Indien'
        WHEN appln_auth = 'BR' THEN 'Brasilien'
        ELSE appln_auth
    END AS authority_name,
    MAX(CASE WHEN period = 'Periode 1 (2014-2018)' THEN applications END) AS apps_2014_2018,
    MAX(CASE WHEN period = 'Periode 2 (2019-2024)' THEN applications END) AS apps_2019_2024,
    MAX(CASE WHEN period = 'Periode 1 (2014-2018)' THEN families END) AS families_2014_2018,
    MAX(CASE WHEN period = 'Periode 2 (2019-2024)' THEN families END) AS families_2019_2024,
    ROUND(
        SAFE_DIVIDE(
            MAX(CASE WHEN period = 'Periode 2 (2019-2024)' THEN applications END) -
            MAX(CASE WHEN period = 'Periode 1 (2014-2018)' THEN applications END),
            MAX(CASE WHEN period = 'Periode 1 (2014-2018)' THEN applications END)
        ) * 100, 1
    ) AS change_percent
FROM geo_by_period
GROUP BY appln_auth
ORDER BY COALESCE(
    MAX(CASE WHEN period = 'Periode 2 (2019-2024)' THEN applications END), 0
) DESC;
""")

df_c.to_csv('airbus_c_geographie.csv', index=False)
df_c

Query took 2.26s (18 rows)


,appln_auth,authority_name,apps_2014_2018,apps_2019_2024,families_2014_2018,families_2019_2024,change_percent
0,US,US Patent Office,3414,3097.0,3329,3018.0,-9.3
1,EP,Europäisches Patentamt,2441,2926.0,2332,2851.0,19.9
2,CN,China (CNIPA),1062,1655.0,1024,1643.0,55.8
3,FR,Frankreich (INPI),1659,1298.0,1650,1292.0,-21.8
4,GB,GB,522,684.0,510,680.0,31.0
5,WO,PCT (WIPO),576,502.0,563,498.0,-12.8
6,DE,Deutschland (DPMA),999,417.0,997,413.0,-58.3
7,CA,CA,394,202.0,388,199.0,-48.7
8,ES,ES,343,194.0,337,193.0,-43.4
9,KR,Südkorea (KIPO),161,90.0,153,88.0,-44.1


# QUERY D: Technology Fields and Shifts

### Business Question
In which technology fields does Airbus patent most heavily? Are there noticeable shifts (e.g., more digitalization, drones, sustainability/hydrogen)?

**Stakeholder:** R&D strategy, technology scouting

**Explanation:** Uses `tls230_appln_techn_field` for mapping to the 35 WIPO technology fields. Compares two periods.

In [5]:
df_d = timed_query("""
WITH airbus_apps AS (
    SELECT DISTINCT
        a.appln_id,
        a.appln_filing_year,
        CASE
            WHEN a.appln_filing_year BETWEEN 2014 AND 2018 THEN 'early'
            WHEN a.appln_filing_year BETWEEN 2019 AND 2024 THEN 'recent'
        END AS period
    FROM tls201_appln a
    JOIN tls207_pers_appln pa ON a.appln_id = pa.appln_id
    JOIN tls206_person p ON pa.person_id = p.person_id
    WHERE pa.applt_seq_nr > 0
      AND a.appln_filing_year BETWEEN 2014 AND 2024
      AND LOWER(p.person_name) LIKE '%airbus%'
),

tech_by_period AS (
    SELECT
        tfi.techn_sector,
        tfi.techn_field,
        tf.techn_field_nr,
        aa.period,
        COUNT(DISTINCT aa.appln_id) AS applications
    FROM airbus_apps aa
    JOIN tls230_appln_techn_field tf ON aa.appln_id = tf.appln_id
    JOIN tls901_techn_field_ipc tfi ON tf.techn_field_nr = tfi.techn_field_nr
    WHERE aa.period IS NOT NULL
    GROUP BY tfi.techn_sector, tfi.techn_field, tf.techn_field_nr, aa.period
)

SELECT
    techn_sector,
    techn_field,
    MAX(CASE WHEN period = 'early' THEN applications END) AS apps_2014_2018,
    MAX(CASE WHEN period = 'recent' THEN applications END) AS apps_2019_2024,
    ROUND(
        SAFE_DIVIDE(
            MAX(CASE WHEN period = 'recent' THEN applications END) -
            MAX(CASE WHEN period = 'early' THEN applications END),
            MAX(CASE WHEN period = 'early' THEN applications END)
        ) * 100, 1
    ) AS growth_percent
FROM tech_by_period
GROUP BY techn_sector, techn_field
HAVING COALESCE(MAX(CASE WHEN period = 'early' THEN applications END), 0) +
       COALESCE(MAX(CASE WHEN period = 'recent' THEN applications END), 0) >= 20
ORDER BY growth_percent DESC NULLS LAST;
""")

df_d.to_csv('airbus_d_technologiefelder.csv', index=False)
df_d

Query took 3.42s (31 rows)


,techn_sector,techn_field,apps_2014_2018,apps_2019_2024,growth_percent
0,Electrical engineering,Basic communication processes,45,77,71.1
1,Electrical engineering,"Electrical machinery, apparatus, energy",681,791,16.2
2,Instruments,Optics,130,144,10.8
3,Instruments,Analysis of biological materials,19,21,10.5
4,Mechanical engineering,"Engines, pumps, turbines",746,790,5.9
5,Mechanical engineering,Mechanical elements,1222,1248,2.1
6,Mechanical engineering,Transport,7210,7184,-0.4
7,Other fields,Civil engineering,202,199,-1.5
8,Other fields,Other consumer goods,155,151,-2.6
9,Mechanical engineering,Thermal processes and apparatus,128,122,-4.7


# QUERY E: IPC Main Classes Top 25 with Trend

### Business Question
Which IPC classes dominate at Airbus and which are growing the fastest? This concretely shows which technologies are being invested in.

**Stakeholder:** Patent department, technology benchmarking

**Explanation:** Extracts IPC main classes (4 characters, e.g., B64C = aircraft) and compares periods. B64 (aeronautics) should dominate, but shifts toward G06 (computing), H04 (communication) indicate digitalization.

In [6]:
df_e = timed_query("""
WITH airbus_apps AS (
    SELECT DISTINCT
        a.appln_id,
        a.appln_filing_year,
        CASE
            WHEN a.appln_filing_year BETWEEN 2014 AND 2018 THEN 'early'
            WHEN a.appln_filing_year BETWEEN 2019 AND 2024 THEN 'recent'
        END AS period
    FROM tls201_appln a
    JOIN tls207_pers_appln pa ON a.appln_id = pa.appln_id
    JOIN tls206_person p ON pa.person_id = p.person_id
    WHERE pa.applt_seq_nr > 0
      AND a.appln_filing_year BETWEEN 2014 AND 2024
      AND LOWER(p.person_name) LIKE '%airbus%'
),

ipc_analysis AS (
    SELECT
        SUBSTR(ipc.ipc_class_symbol, 1, 4) AS ipc_main_class,
        aa.period,
        COUNT(DISTINCT aa.appln_id) AS applications
    FROM airbus_apps aa
    JOIN tls209_appln_ipc ipc ON aa.appln_id = ipc.appln_id
    WHERE aa.period IS NOT NULL
    GROUP BY SUBSTR(ipc.ipc_class_symbol, 1, 4), aa.period
)

SELECT
    ipc_main_class,
    CASE ipc_main_class
        WHEN 'B64C' THEN 'Flugzeuge/Hubschrauber'
        WHEN 'B64D' THEN 'Flugzeugausrüstung'
        WHEN 'B64F' THEN 'Flughafeneinrichtungen'
        WHEN 'B64G' THEN 'Kosmonautik'
        WHEN 'F02C' THEN 'Gasturbinen'
        WHEN 'F02K' THEN 'Strahltriebwerke'
        WHEN 'G01S' THEN 'Radar/Navigation'
        WHEN 'G06F' THEN 'Datenverarbeitung'
        WHEN 'G06N' THEN 'KI/Neuronale Netze'
        WHEN 'H04L' THEN 'Datenübertragung'
        WHEN 'H04B' THEN 'Nachrichtentechnik'
        WHEN 'B29C' THEN 'Kunststoffverarbeitung (Composite)'
        WHEN 'G05B' THEN 'Steuerungstechnik'
        WHEN 'G05D' THEN 'Regelungstechnik'
        WHEN 'H01Q' THEN 'Antennen'
        WHEN 'B32B' THEN 'Schichtwerkstoffe'
        WHEN 'F16B' THEN 'Befestigungselemente'
        WHEN 'C08J' THEN 'Polymer-Verarbeitung'
        WHEN 'H02J' THEN 'Energieverteilung'
        WHEN 'Y02T' THEN 'Nachhaltiger Transport'
        ELSE ipc_main_class
    END AS description,
    COALESCE(MAX(CASE WHEN period = 'early' THEN applications END), 0) AS apps_2014_2018,
    COALESCE(MAX(CASE WHEN period = 'recent' THEN applications END), 0) AS apps_2019_2024,
    COALESCE(MAX(CASE WHEN period = 'early' THEN applications END), 0) +
    COALESCE(MAX(CASE WHEN period = 'recent' THEN applications END), 0) AS total,
    ROUND(
        SAFE_DIVIDE(
            MAX(CASE WHEN period = 'recent' THEN applications END) -
            MAX(CASE WHEN period = 'early' THEN applications END),
            MAX(CASE WHEN period = 'early' THEN applications END)
        ) * 100, 1
    ) AS growth_percent
FROM ipc_analysis
GROUP BY ipc_main_class
HAVING COALESCE(MAX(CASE WHEN period = 'early' THEN applications END), 0) +
       COALESCE(MAX(CASE WHEN period = 'recent' THEN applications END), 0) >= 20
ORDER BY total DESC
LIMIT 25;
""")

df_e.to_csv('airbus_e_ipc_klassen.csv', index=False)
df_e

Query took 4.90s (25 rows)


,ipc_main_class,description,apps_2014_2018,apps_2019_2024,total,growth_percent
0,B64C,Flugzeuge/Hubschrauber,3995,3575,7570,-10.5
1,B64D,Flugzeugausrüstung,3323,3757,7080,13.1
2,B29C,Kunststoffverarbeitung (Composite),1100,653,1753,-40.6
3,B64F,Flughafeneinrichtungen,700,921,1621,31.6
4,B64G,Kosmonautik,484,324,808,-33.1
5,G06F,Datenverarbeitung,482,301,783,-37.6
6,F02C,Gasturbinen,321,459,780,43.0
7,F16B,Befestigungselemente,366,335,701,-8.5
8,B32B,Schichtwerkstoffe,422,257,679,-39.1
9,G05D,Regelungstechnik,410,254,664,-38.0


# QUERY F: Grant Rate and Time-to-Grant

### Business Question
How successful is Airbus in patent prosecution? How long does it take to obtain a grant at different offices?

**Stakeholder:** Patent prosecution, portfolio management

**Explanation:** Uses `publn_first_grant = 'Y'` (more reliable than legal events). Time-to-grant = first publication of the grant minus filing date.

In [7]:
df_f = timed_query("""
WITH airbus_apps AS (
    SELECT DISTINCT
        a.appln_id,
        a.appln_auth,
        a.appln_filing_date,
        a.appln_filing_year,
        a.granted
    FROM tls201_appln a
    JOIN tls207_pers_appln pa ON a.appln_id = pa.appln_id
    JOIN tls206_person p ON pa.person_id = p.person_id
    WHERE pa.applt_seq_nr > 0
      AND a.appln_filing_year BETWEEN 2014 AND 2021  -- bis 2021 für genug Grant-Zeit
      AND LOWER(p.person_name) LIKE '%airbus%'
      AND a.appln_auth IN ('EP', 'US', 'CN', 'FR', 'DE', 'JP', 'KR')
),

grant_info AS (
    SELECT
        aa.appln_id,
        aa.appln_auth,
        aa.appln_filing_date,
        aa.appln_filing_year,
        aa.granted,
        MIN(CASE WHEN pub.publn_first_grant = 'Y' THEN pub.publn_date END) AS grant_date
    FROM airbus_apps aa
    LEFT JOIN tls211_pat_publn pub ON aa.appln_id = pub.appln_id
    GROUP BY aa.appln_id, aa.appln_auth, aa.appln_filing_date, aa.appln_filing_year, aa.granted
)

SELECT
    appln_auth,
    COUNT(*) AS total_applications,
    COUNT(CASE WHEN granted = 'Y' THEN 1 END) AS granted,
    ROUND(COUNT(CASE WHEN granted = 'Y' THEN 1 END) * 100.0 / COUNT(*), 1) AS grant_rate_pct,
    ROUND(AVG(
        CASE WHEN grant_date IS NOT NULL AND grant_date > appln_filing_date
        THEN DATE_DIFF(grant_date, appln_filing_date, DAY) / 365.25
        END
    ), 1) AS avg_years_to_grant,
    ROUND(APPROX_QUANTILES(
        CASE WHEN grant_date IS NOT NULL AND grant_date > appln_filing_date
        THEN DATE_DIFF(grant_date, appln_filing_date, DAY) / 365.25
        END, 2)[OFFSET(1)], 1) AS median_years_to_grant
FROM grant_info
GROUP BY appln_auth
ORDER BY total_applications DESC;
""")

df_f.to_csv('airbus_f_erteilungsquote.csv', index=False)
df_f

Query took 7.29s (7 rows)


,appln_auth,total_applications,granted,grant_rate_pct,avg_years_to_grant,median_years_to_grant
0,US,4953,4050,81.8,2.8,2.7
1,EP,3872,2905,75.0,3.3,3.0
2,FR,2461,1895,77.0,2.8,2.4
3,CN,1802,868,48.2,4.0,4.0
4,DE,1362,318,23.3,3.8,3.0
5,KR,187,138,73.8,2.5,1.9
6,JP,128,64,50.0,2.9,2.4


# QUERY G: Co-Applicants and Cooperation Strategy

### Business Question
With whom does Airbus cooperate on patent filings? Are there trends toward more or fewer co-filings?

**Stakeholder:** Open innovation, research cooperations

**Explanation:** Identifies co-applicants on Airbus patents. `nb_applicants > 1` indicates co-filings.

In [8]:
df_g = timed_query("""
WITH airbus_coapplications AS (
    SELECT DISTINCT
        a.appln_id,
        a.appln_filing_year,
        a.nb_applicants,
        p.person_name AS co_applicant,
        p.psn_sector AS co_sector,
        p.person_ctry_code AS co_country
    FROM tls201_appln a
    JOIN tls207_pers_appln pa ON a.appln_id = pa.appln_id
    JOIN tls206_person p ON pa.person_id = p.person_id
    WHERE pa.applt_seq_nr > 0
      AND a.nb_applicants > 1
      AND a.appln_filing_year BETWEEN 2014 AND 2024
      AND NOT LOWER(p.person_name) LIKE '%airbus%'  -- Nur den Partner zeigen
      AND a.appln_id IN (
          -- Nur Anmeldungen wo Airbus auch Anmelder ist
          SELECT pa2.appln_id
          FROM tls207_pers_appln pa2
          JOIN tls206_person p2 ON pa2.person_id = p2.person_id
          WHERE pa2.applt_seq_nr > 0
            AND LOWER(p2.person_name) LIKE '%airbus%'
      )
)

SELECT
    co_applicant,
    co_sector,
    co_country,
    COUNT(DISTINCT appln_id) AS joint_applications,
    MIN(appln_filing_year) AS first_cooperation,
    MAX(appln_filing_year) AS last_cooperation,
    COUNT(DISTINCT appln_filing_year) AS active_years
FROM airbus_coapplications
GROUP BY co_applicant, co_sector, co_country
HAVING COUNT(DISTINCT appln_id) >= 3
ORDER BY joint_applications DESC
LIMIT 30;
""")

df_g.to_csv('airbus_g_kooperationen.csv', index=False)
df_g

Query took 4.29s (30 rows)


,co_applicant,co_sector,co_country,joint_applications,first_cooperation,last_cooperation,active_years
0,ARIANEGROUP SAS,COMPANY,FR,30,2014,2023,7
1,Centre National de la Recherche Scientifique,GOV NON-PROFIT,FR,22,2014,2024,6
2,CENTRE NATIONAL D'ETUDES SPATIALES CNES,GOV NON-PROFIT,FR,21,2014,2024,9
3,Fraunhofer-Gesellschaft zur Förderung der ange...,GOV NON-PROFIT,DE,16,2014,2024,10
4,DEUTSCHES ZENTRUM FÜR LUFT- UND RAUMFAHRT E.V.,GOV NON-PROFIT,DE,13,2014,2023,8
5,EADS Deutschland GmbH,COMPANY,DE,13,2014,2015,2
6,University of Surrey,UNIVERSITY,GB,11,2014,2020,4
7,Centre National d'Etudes Spatiales (CNES),GOV NON-PROFIT,FR,11,2014,2020,5
8,EUROPEAN AERONAUTIC DEFENCE AND SPACE COMPANY ...,COMPANY,FR,11,2014,2015,2
9,Bombardier Inc.,COMPANY,CA,11,2014,2017,4


# QUERY H: Patent Family Size as Strategy Indicator

### Business Question
How broadly does Airbus protect its inventions internationally? Is the average family size increasing (= more countries per invention)?

**Stakeholder:** IP budget planning, internationalization

**Explanation:** The DOCDB family size shows in how many countries an invention is protected. Larger families = higher strategic importance.

In [9]:
df_h = timed_query("""
WITH airbus_families AS (
    SELECT DISTINCT
        a.docdb_family_id,
        a.appln_filing_year,
        a.docdb_family_size
    FROM tls201_appln a
    JOIN tls207_pers_appln pa ON a.appln_id = pa.appln_id
    JOIN tls206_person p ON pa.person_id = p.person_id
    WHERE pa.applt_seq_nr > 0
      AND a.appln_filing_year BETWEEN 2014 AND 2024
      AND LOWER(p.person_name) LIKE '%airbus%'
      AND a.docdb_family_id > 0
      -- Nur Erstanmeldung pro Familie für korrekte Zuordnung
      AND a.appln_id = a.earliest_filing_id
)

SELECT
    appln_filing_year,
    COUNT(DISTINCT docdb_family_id) AS unique_families,
    ROUND(AVG(docdb_family_size), 1) AS avg_family_size,
    ROUND(APPROX_QUANTILES(docdb_family_size, 2)[OFFSET(1)], 0) AS median_family_size,
    MAX(docdb_family_size) AS max_family_size,
    COUNT(CASE WHEN docdb_family_size >= 10 THEN 1 END) AS large_families_10plus,
    COUNT(CASE WHEN docdb_family_size >= 20 THEN 1 END) AS very_large_families_20plus,
    ROUND(COUNT(CASE WHEN docdb_family_size >= 10 THEN 1 END) * 100.0 /
          COUNT(DISTINCT docdb_family_id), 1) AS pct_large_families
FROM airbus_families
GROUP BY appln_filing_year
ORDER BY appln_filing_year;
""")

df_h.to_csv('airbus_h_familiengroesse.csv', index=False)
df_h

Query took 4.25s (11 rows)


,appln_filing_year,unique_families,avg_family_size,median_family_size,max_family_size,large_families_10plus,very_large_families_20plus,pct_large_families
0,2014,861,2.8,2.0,15,5,0,0.6
1,2015,910,2.8,3.0,12,2,0,0.2
2,2016,907,2.4,2.0,11,2,0,0.2
3,2017,731,2.8,3.0,16,2,0,0.3
4,2018,698,3.1,3.0,19,3,0,0.4
5,2019,827,2.5,2.0,11,2,0,0.2
6,2020,555,2.8,3.0,11,1,0,0.2
7,2021,611,2.7,3.0,7,0,0,0.0
8,2022,629,2.7,3.0,6,0,0,0.0
9,2023,680,2.3,2.0,6,0,0,0.0


# QUERY I: Sustainability / Future Technology Patents

### Business Question
How heavily is Airbus investing in future technologies? Specifically: hydrogen, electric flight, AI, drones/UAV, additive manufacturing

**Stakeholder:** Technology strategy, sustainability, ESG reporting

**Explanation:** Searches for specific CPC/IPC classes **AND** title keywords indicating future technologies. CPC Y02T = Climate Change Mitigation - Transport

In [10]:
df_i = timed_query("""
WITH airbus_apps AS (
    SELECT DISTINCT
        a.appln_id,
        a.appln_filing_year,
        a.docdb_family_id
    FROM tls201_appln a
    JOIN tls207_pers_appln pa ON a.appln_id = pa.appln_id
    JOIN tls206_person p ON pa.person_id = p.person_id
    WHERE pa.applt_seq_nr > 0
      AND a.appln_filing_year BETWEEN 2014 AND 2024
      AND LOWER(p.person_name) LIKE '%airbus%'
),

future_tech AS (
    SELECT
        aa.appln_id,
        aa.appln_filing_year,
        CASE
            -- Wasserstoff/Brennstoffzelle
            WHEN cpc.cpc_class_symbol LIKE 'Y02E%60/5%'
              OR cpc.cpc_class_symbol LIKE 'H01M%8/%'
              OR cpc.cpc_class_symbol LIKE 'C01B%3/%'
              OR LOWER(t.appln_title) LIKE '%hydrogen%'
              OR LOWER(t.appln_title) LIKE '%fuel cell%'
              OR LOWER(t.appln_title) LIKE '%wasserstoff%'
            THEN 'Wasserstoff/Brennstoffzelle'
            -- Elektrischer Antrieb
            WHEN cpc.cpc_class_symbol LIKE 'B64D%27/24%'
              OR cpc.cpc_class_symbol LIKE 'H02K%'
              OR (LOWER(t.appln_title) LIKE '%electric%' AND LOWER(t.appln_title) LIKE '%propuls%')
              OR LOWER(t.appln_title) LIKE '%hybrid%propuls%'
            THEN 'Elektro-/Hybridantrieb'
            -- KI/Machine Learning
            WHEN cpc.cpc_class_symbol LIKE 'G06N%'
              OR LOWER(t.appln_title) LIKE '%machine learning%'
              OR LOWER(t.appln_title) LIKE '%neural network%'
              OR LOWER(t.appln_title) LIKE '%deep learning%'
              OR LOWER(t.appln_title) LIKE '%artificial intell%'
            THEN 'Künstliche Intelligenz'
            -- UAV/Drohnen
            WHEN LOWER(t.appln_title) LIKE '%unmanned%'
              OR LOWER(t.appln_title) LIKE '%uav%'
              OR LOWER(t.appln_title) LIKE '%drone%'
              OR LOWER(t.appln_title) LIKE '%urban air mobil%'
              OR LOWER(t.appln_title) LIKE '%evtol%'
            THEN 'UAV/Drohnen/Urban Air Mobility'
            -- Additive Fertigung
            WHEN cpc.cpc_class_symbol LIKE 'B33Y%'
              OR LOWER(t.appln_title) LIKE '%additive manufactur%'
              OR LOWER(t.appln_title) LIKE '%3d print%'
            THEN 'Additive Fertigung (3D-Druck)'
            -- Nachhaltigkeit allgemein
            WHEN cpc.cpc_class_symbol LIKE 'Y02T%'
            THEN 'Nachhaltiger Transport (Y02T)'
        END AS future_tech_area
    FROM airbus_apps aa
    LEFT JOIN tls224_appln_cpc cpc ON aa.appln_id = cpc.appln_id
    LEFT JOIN tls202_appln_title t ON aa.appln_id = t.appln_id
    WHERE t.appln_title_lg = 'en'  -- Englische Titel für Keyword-Suche
)

SELECT
    future_tech_area,
    appln_filing_year,
    COUNT(DISTINCT appln_id) AS applications
FROM future_tech
WHERE future_tech_area IS NOT NULL
GROUP BY future_tech_area, appln_filing_year
ORDER BY future_tech_area, appln_filing_year;
""")

df_i.to_csv('airbus_i_zukunftstech.csv', index=False)
df_i

Query took 9.21s (65 rows)


,future_tech_area,appln_filing_year,applications
0,Additive Fertigung (3D-Druck),2014,28
1,Additive Fertigung (3D-Druck),2015,40
2,Additive Fertigung (3D-Druck),2016,70
3,Additive Fertigung (3D-Druck),2017,39
4,Additive Fertigung (3D-Druck),2018,53
...,...,...,...
60,Wasserstoff/Brennstoffzelle,2020,43
61,Wasserstoff/Brennstoffzelle,2021,39
62,Wasserstoff/Brennstoffzelle,2022,111
63,Wasserstoff/Brennstoffzelle,2023,112


# QUERY J: Inventor Locations (NUTS Regions)

### Business Question
Where are Airbus's inventors located? How is the innovation capacity distributed across sites (Toulouse, Hamburg, Munich, Bremen, Manching...)?

**Stakeholder:** Site policy, research funding, policy advisory

**Explanation:** Uses NUTS codes of inventors (`invt_seq_nr > 0`). NUTS Level 2 gives the region, Level 3 the district.

In [11]:
df_j = timed_query("""
SELECT
    p.person_ctry_code AS inventor_country,
    SUBSTR(p.nuts, 1, 4) AS nuts_region,
    n.nuts_label AS region_name,
    COUNT(DISTINCT a.appln_id) AS applications,
    COUNT(DISTINCT p.person_id) AS unique_inventors,
    CASE
        WHEN a.appln_filing_year BETWEEN 2014 AND 2018 THEN 'early'
        WHEN a.appln_filing_year BETWEEN 2019 AND 2024 THEN 'recent'
    END AS period
FROM tls201_appln a
JOIN tls207_pers_appln pa ON a.appln_id = pa.appln_id
JOIN tls206_person p ON pa.person_id = p.person_id
LEFT JOIN tls904_nuts n ON SUBSTR(p.nuts, 1, 4) = n.nuts
WHERE pa.invt_seq_nr > 0  -- Nur Erfinder, nicht Anmelder
  AND a.appln_filing_year BETWEEN 2014 AND 2024
  AND a.appln_id IN (
      SELECT pa2.appln_id
      FROM tls207_pers_appln pa2
      JOIN tls206_person p2 ON pa2.person_id = p2.person_id
      WHERE pa2.applt_seq_nr > 0
        AND LOWER(p2.person_name) LIKE '%airbus%'
  )
  AND p.nuts IS NOT NULL
  AND LENGTH(p.nuts) >= 4
GROUP BY p.person_ctry_code, SUBSTR(p.nuts, 1, 4), n.nuts_label,
         CASE
             WHEN a.appln_filing_year BETWEEN 2014 AND 2018 THEN 'early'
             WHEN a.appln_filing_year BETWEEN 2019 AND 2024 THEN 'recent'
         END
HAVING COUNT(DISTINCT a.appln_id) >= 10
ORDER BY applications DESC;
""")

df_j.to_csv('airbus_j_erfinder_standorte.csv', index=False)
df_j

Query took 5.23s (42 rows)


,inventor_country,nuts_region,region_name,applications,unique_inventors,period
0,FR,FRJ2,None,948,1291,recent
1,DE,DE60,Hamburg,893,781,recent
2,DE,DE60,Hamburg,654,731,early
3,DE,DE21,Oberbayern,410,547,early
4,GB,UKK5,None,390,507,recent
5,DE,DE21,Oberbayern,336,403,recent
6,ES,ES30,Comunidad de Madrid,275,718,early
7,ES,ES30,Comunidad de Madrid,254,504,recent
8,FR,FR62,Midi-Pyrénées,229,425,early
9,FR,FR82,Provence-Alpes-Côte d'Azur,224,298,early


---

# Summary: Expected Results and Interpretation Guidelines

The 10 queries together provide a complete picture of the Airbus patent strategy:

| Query | Focus | Description |
|-------|-------|-------------|
| **A** | Name variants | Foundation – shows all Airbus entities in PATSTAT |
| **B** | Filing trend | Big picture – patent activity by business division/year |
| **C** | Geography | Where is protection sought – EP/FR/US/CN distribution and trends |
| **D** | Technology fields | What is protected – WIPO 35 technology fields |
| **E** | IPC classes | Detail – top 25 IPC classes with trends (B64=aero, G06=IT...) |
| **F** | Grant rate | How successful – grant rate and time-to-grant per office |
| **G** | Cooperations | With whom – joint applications with partners |
| **H** | Family size | How broad – international scope of protection per invention |
| **I** | Future tech | Where is it heading – H2, e-flight, AI, drones, 3D printing |
| **J** | Inventor locations | Where does innovation originate – NUTS regions |

### Typical Hypotheses to Validate

1. Airbus is shifting IP investments from traditional aircraft manufacturing to digitalization (G06) and sustainability (Y02T)
2. The share of China filings is growing disproportionately
3. Airbus Defence & Space is growing faster than Airbus Operations
4. Cooperations with universities are increasing
5. Toulouse/FR dominates for inventors, but Hamburg/DE is growing
6. Family size is increasing (= more selective but broader filings)